In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ["HF_HOME"] = "/content/drive/MyDrive/huggingface_cache"

!pip install -q vllm lxml

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile annotator.py
import argparse, json, os, re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from lxml import etree
from vllm import LLM, SamplingParams

# --- A100 Configuration ---
MODEL_ID = os.getenv("MODEL_ID", "openai/gpt-oss-20b")
GPU_UTIL = float(os.getenv("GPU_UTIL", "0.90"))
MAX_MODEL_LEN = int(os.getenv("MAX_MODEL_LEN", "16384"))
DTYPE = os.getenv("DTYPE", "bfloat16")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "2000")) # Doubled so it doesn't get cut off while thinking
SUMMARY_WORD_LIMIT = int(os.getenv("SUMMARY_WORD_LIMIT", "50"))
TP_SIZE = int(os.getenv("VLLM_TP", "1"))
K_TARGET = 1
N_NEIGH = 20

_GLOBAL_LLM: Optional[LLM] = None

# --- Prompt Engine & Few Shots ---
FEWSHOTS_BLOCK = """
EXAMPLES (for reference only)

EXAMPLE A — Starting a backup subtask (depth=-1)
neighbor_tail:
  - id=0 depth=0  summary="List project directory contents"
  - id=1 depth=0  summary="Inspect size of source and data folders"
currDepth: 0
input xml:
<event>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>-</user_input><system_output>-</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>z</user_input><system_output>z</system_output>
  <user_input>f</user_input><system_output>f</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>b</user_input><system_output>b</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>k</user_input><system_output>k</system_output>
  <user_input>u</user_input><system_output>u</system_output>
  <user_input>p</user_input><system_output>p</system_output>
  <user_input>.</user_input><system_output>.</system_output>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>s</user_input><system_output>s</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <system_output>Creating backup.tar...</system_output>
</event>
output:
{"annotation": "Create compressed backup archive of source data", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE B — Continuing backup subtask (depth=0)
neighbor_tail:
  - id=2 depth=-1 summary="Create compressed backup archive of source data"
currDepth: -1
input xml:
<event>
  <user_input>l</user_input><system_output>l</system_output>
  <user_input>s</user_input><system_output>s</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>-</user_input><system_output>-</system_output>
  <user_input>l</user_input><system_output>l</system_output>
  <user_input>h</user_input><system_output>h</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>b</user_input><system_output>b</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>k</user_input><system_output>k</system_output>
  <user_input>u</user_input><system_output>u</system_output>
  <user_input>p</user_input><system_output>p</system_output>
  <user_input>.</user_input><system_output>.</system_output>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <system_output>-rw-r--r-- 1 user staff 42M backup.tar</system_output>
</event>
output:
{"annotation": "Verify backup archive and check file size", "depth": 0}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE C — Finishing backup subtask (depth=+1)
neighbor_tail:
  - id=2 depth=-1 summary="Create compressed backup archive of source data"
  - id=3 depth=0  summary="Verify backup archive and check file size"
currDepth: -1
input xml:
<event>
  <user_input>m</user_input><user_input>v</user_input><user_input> </user_input>
  <user_input>b</user_input><user_input>a</user_input><user_input>c</user_input>
  <system_output>Moved to archive/</system_output>
</event>
output:
{"annotation": "Move backup to archive folder and complete backup task", "depth": 1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE D — Starting test/debug subtask (depth=-1)
currDepth: 0
input xml:
<event>
  <user_input>p</user_input><user_input>y</user_input><user_input>t</user_input><user_input>e</user_input><user_input>s</user_input><user_input>t</user_input>
  <system_output>===== test session starts =====</system_output>
</event>
output:
{"annotation": "Start pytest test run for project", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE E — Nested editor within environment setup (depth=-1)
currDepth: -1
input xml:
<event>
  <user_input>v</user_input><user_input>i</user_input><user_input>m</user_input>
  <system_output>Opening vim...</system_output>
</event>
output:
{"annotation": "Open config file in vim during environment setup", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE F — Exit editor, stay in parent task (depth=+1)
currDepth: -2
input xml:
<event>
  <user_input>:</user_input><user_input>w</user_input><user_input>q</user_input>
  <system_output>(venv) $</system_output>
</event>
output:
{"annotation": "Save config changes and exit vim", "depth": 1}
"""

@dataclass
class Event:
    idx: int
    xml: str
    depth_xml: Optional[int] = None
    summary_xml: Optional[str] = None

events: List[Event] = []
pred: Dict[int, Dict[str, object]] = {}

def load_events(xml_path: str) -> List[Event]:
    tree = etree.parse(xml_path)
    root = tree.getroot()
    out = []
    for i, ev_el in enumerate(root.xpath("//event")):
        xml_str = etree.tostring(ev_el, encoding="unicode")
        out.append(Event(idx=i, xml=xml_str))
    return out

def compute_curr_depth_upto(idx: int) -> int:
    curr = 0
    for i in range(idx):
        dep = events[i].depth_xml
        if dep is None: continue
        if dep == -1: curr -= 1
        elif dep > 0: curr += dep
    return curr

def make_flush_package(upto_idx: int, K: int = 1, N: int = 10) -> Dict:
    target_idxs = list(range(max(0, upto_idx - K + 1), upto_idx + 1))
    start_neigh = max(0, target_idxs[0] - N)
    neighbor_idxs = list(range(start_neigh, target_idxs[0]))

    def get_sum(i): return events[i].summary_xml if (0 <= i < len(events) and events[i].summary_xml) else "???"
    def get_dep(i): return events[i].depth_xml if (0 <= i < len(events) and events[i].depth_xml is not None) else 999

    neighbor_info = [f"- id={i} depth={get_dep(i)}  summary={get_sum(i)}" for i in neighbor_idxs]
    target_events = [events[i].xml for i in target_idxs if 0 <= i < len(events)]

    return {
        "target_idxs": target_idxs,
        "neighbor_info": neighbor_info,
        "target_events": target_events,
        "currDepth": compute_curr_depth_upto(target_idxs[0]),
    }

def build_instruction(pkg: dict) -> str:
    # --- NEW: AGGRESSIVE MASSIVE EVENT TRUNCATION ---
    MAX_CHARS = 3000 # Roughly 750 tokens
    truncated_targets = []

    for i, xml in zip(pkg["target_idxs"], pkg["target_events"]):
        if len(xml) > MAX_CHARS:
            # Keep the first 1000 chars (the command) and last 2000 chars (the final result)
            snipped_xml = (
                xml[:1000] +
                f"\n\n... [ ✂️ SYSTEM OUTPUT TRUNCATED - HIDDEN {len(xml) - MAX_CHARS} CHARACTERS ] ...\n\n" +
                xml[-2000:]
            )
            truncated_targets.append(f'<target id="{i}">\n{snipped_xml}\n</target>')
        else:
            truncated_targets.append(f'<target id="{i}">\n{xml}\n</target>')

    targets_xml = "\n".join(truncated_targets)

    return f"""<task>
You are an expert terminal session annotator. Identify goals/subgoals and generate concise action summaries.
</task>

<think_first>
- Keep reasoning CONCISE and FOCUSED
- In <think>...</think>: analyze the command, check depth logic, then conclude
- Aim for 2-3 sentences of reasoning maximum
- Skip obvious observations
- Use neighbors ONLY for continuity; do not invent context.
</think_first>

<rules>
- the user's keystrokes appear separately; combine them to form the full command before interpreting it
- depth is an integer (≥ -1); -1 for subevent (new task started), 0 for same level (still doing the same task), >0 to exit levels (ended one or multiple tasks)
- maintain stack invariant: currDepth ≤ 0; if depth == -1 then currDepth -= 1; if depth > 0 then currDepth += depth
- write action-oriented summaries; avoid "user", "they", "typed", "inputs", "enters a command
- depth is relative to the previous events and nothing else"
</rules>

<output_format>
You MUST wrap your reasoning in <think>...</think> tags.
After the closing </think> tag, you MUST output EXACTLY ONE valid JSON object on a new line. Do not output anything after the JSON.
{{"annotation": "<action summary ≤{SUMMARY_WORD_LIMIT} words>", "depth": <integer ≥ -1>}}
</output_format>

DEPTH SEMANTICS:
- depth = -1: STARTING a new subtask (entering deeper level)
- depth = 0:  CONTINUING at same level (ongoing work)
- depth = +1: FINISHING a subtask (returning to parent level)

<examples>
{FEWSHOTS_BLOCK}
</examples>

<inputs>
  <curr_depth>{pkg.get("currDepth", 0)}</curr_depth>
  <target_events>\n{targets_xml}\n  </target_events>
</inputs>"""

def load_model() -> LLM:
    global _GLOBAL_LLM
    if _GLOBAL_LLM is not None: return _GLOBAL_LLM
    _GLOBAL_LLM = LLM(
        model=MODEL_ID, tensor_parallel_size=TP_SIZE, gpu_memory_utilization=GPU_UTIL,
        max_model_len=MAX_MODEL_LEN, trust_remote_code=True, dtype=DTYPE
    )
    return _GLOBAL_LLM

def generate_with_thinking(llm: LLM, messages: List[Dict[str, str]]):
    tokenizer = llm.get_tokenizer()
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, repetition_penalty=1.2)
    outputs = llm.generate([prompt], sampling_params)
    text = outputs[0].outputs[0].text.strip()
    return text, text, 0, 0

def parse_depth_summary_pairs(text: str) -> List[Tuple[int, str]]:
    dec = json.JSONDecoder()
    out = []

    # NEW PARSING LOGIC: Strip away everything before and inside the <think> tags
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()

    i = 0
    while True:
        start = text.find("{", i)
        if start == -1: break
        try:
            obj, end = dec.raw_decode(text, start)
            if isinstance(obj, dict) and "depth" in obj and "annotation" in obj:
                out.append((int(obj["depth"]), str(obj["annotation"])))
            i = end
        except:
            i = start + 1
    return out

def run_flushes(evs: List[Event]) -> Dict:
    global events, pred
    events = evs
    llm = load_model()

    for upto in range(len(events)):
        success = False
        text = ""
        pkg = {}

        # DYNAMIC CONTEXT SHRINKING: Step down the neighbors if the token limit is exceeded
        for attempt_N in [10, 5, 2, 0]:
            pkg = make_flush_package(upto, N=attempt_N)
            instr = build_instruction(pkg)

            try:
                _, text, _, _ = generate_with_thinking(llm, [{"role": "user", "content": instr}])
                success = True
                break  # Inference succeeded, exit the retry loop
            except Exception as e:
                # Catch the context length error and try again with fewer neighbors
                if "context length" in str(e).lower() or "vllmvalidationerror" in str(type(e)).lower():
                    print(f"\n⚠️ EVENT {upto} EXCEEDED 16K TOKENS (N={attempt_N}). Shrinking history and retrying...")
                    continue
                else:
                    raise e  # If it's a completely different error, crash normally

        if not success:
            print(f"\n❌ EVENT {upto} IS TOO LARGE EVEN WITH 0 NEIGHBORS. Falling back to default.")
            text = '{"annotation": "Event payload too large for context window.", "depth": 0}'

        print(f"\n--- RAW AI OUTPUT FOR EVENT {upto} ---")
        print(text)
        print("--------------------------------------\n")

        pairs = parse_depth_summary_pairs(text)
        if pairs:
            depth, ann = pairs[0]
            pred[pkg["target_idxs"][0]] = {"depth": depth, "summary": ann}
            events[pkg["target_idxs"][0]].depth_xml = depth
            events[pkg["target_idxs"][0]].summary_xml = ann
            print(f"✅ SUCCESSFULLY PARSED -> Depth: {depth} | {ann}")
        else:
            print(f"❌ FAILED TO PARSE JSON FROM EVENT {upto}")

    return pred

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("input_xml")
    parser.add_argument("output_path")
    args = parser.parse_args()

    evs = load_events(args.input_xml)
    preds = run_flushes(evs)

    with open(args.output_path, "w") as f:
        for idx, v in preds.items():
            f.write(json.dumps({"idx": idx, **v}) + "\n")
    print(f"\nSaved to {args.output_path}")

if __name__ == "__main__":
    main()

Overwriting annotator.py


In [ ]:
import os
from google.colab import files

print("Select your XML file to upload:")
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
os.environ["INPUT_XML"] = input_filename
print(f"Successfully uploaded {input_filename}")

Select your XML file to upload:


Saving renee_rec2_parsed.xml to renee_rec2_parsed.xml
Successfully uploaded renee_rec2_parsed.xml


In [ ]:
import os
from google.colab import files

input_filename = os.environ.get("INPUT_XML")

# Run the script via command line
!python annotator.py {input_filename} output.jsonl

# Trigger browser download of the results
files.download('output.jsonl')

2026-03-16 15:35:01.142648: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-16 15:35:01.161440: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773675301.186058   35251 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773675301.193318   35251 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773675301.210738   35251 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>